## **Etape 1 - Perimetre et geocoding des 35 villes de France**

Objectif:
1. Definir le perimetre des 35 villes imposees.
2. Recuperer leurs coordonnees GPS (latitude/longitude) avec l'API Nominatim.
3. Sauvegarder le resultat dans `outputs/data/cities_geocoded.csv`.

### **Importer les librairies**

In [1]:
from pathlib import Path
import time
import requests
import pandas as pd

### **Definir le perimetre des villes**

In [1]:
cities = [
    "Mont Saint Michel",
    "St Malo",
    "Bayeux",
    "Le Havre",
    "Rouen",
    "Paris",
    "Amiens",
    "Lille",
    "Strasbourg",
    "Chateau du Haut Koenigsbourg",
    "Colmar",
    "Eguisheim",
    "Besancon",
    "Dijon",
    "Annecy",
    "Grenoble",
    "Lyon",
    "Gorges du Verdon",
    "Bormes les Mimosas",
    "Cassis",
    "Marseille",
    "Aix en Provence",
    "Avignon",
    "Uzes",
    "Nimes",
    "Aigues Mortes",
    "Saintes Maries de la mer",
    "Collioure",
    "Carcassonne",
    "Ariege",
    "Toulouse",
    "Montauban",
    "Biarritz",
    "Bayonne",
    "La Rochelle",
]

print(f"Nombre de villes dans le perimetre: {len(cities)}")

Nombre de villes dans le perimetre: 35


### **Definir la fonction de geocoding**

- construit la requete Nominatim,
- envoie la requete HTTP avec un `User-Agent`,
- extrait latitude/longitude et metadonnees,
- renvoie une structure standard, meme en cas de ville non trouvee.

In [3]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (contact: raphael@nexthope.net)"
}


def geocode_city(city_name: str, country: str = "France") -> dict:
    query = f"{city_name}, {country}"
    params = {
        "q": query,
        "format": "jsonv2",
        "limit": 1,
        "addressdetails": 1,
    }
    response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if not payload:
        return {
            "city_name": city_name,
            "query": query,
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_type": None,
            "osm_id": None,
            "class": None,
            "type": None,
            "found": False,
        }

    item = payload[0]
    return {
        "city_name": city_name,
        "query": query,
        "latitude": float(item.get("lat")),
        "longitude": float(item.get("lon")),
        "display_name": item.get("display_name"),
        "osm_type": item.get("osm_type"),
        "osm_id": item.get("osm_id"),
        "class": item.get("class"),
        "type": item.get("type"),
        "found": True,
    }

### **Lancer le geocoding pour les 35 villes**

- appelle Nominatim pour chaque ville,
- gere les erreurs sans interrompre tout le pipeline,
- applique une pause d'1 seconde entre les requetes,
- construit un DataFrame `cities_geo_df` avec un `city_id`.

In [4]:
records = []

for idx, city in enumerate(cities, start=1):
    try:
        rec = geocode_city(city)
        status = "OK" if rec["found"] else "NOT_FOUND"
    except Exception as exc:
        rec = {
            "city_name": city,
            "query": f"{city}, France",
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_type": None,
            "osm_id": None,
            "class": None,
            "type": None,
            "found": False,
            "error": str(exc),
        }
        status = "ERROR"

    records.append(rec)
    print(f"[{idx:02d}/{len(cities)}] {city}: {status}")

    # Bonne pratique Nominatim: limiter la frequence de requetes
    time.sleep(1)

cities_geo_df = pd.DataFrame(records)
cities_geo_df.insert(0, "city_id", range(1, len(cities_geo_df) + 1))

cities_geo_df.head()

[01/35] Mont Saint Michel: OK
[02/35] St Malo: OK
[03/35] Bayeux: OK
[04/35] Le Havre: OK
[05/35] Rouen: OK
[06/35] Paris: OK
[07/35] Amiens: OK
[08/35] Lille: OK
[09/35] Strasbourg: OK
[10/35] Chateau du Haut Koenigsbourg: OK
[11/35] Colmar: OK
[12/35] Eguisheim: OK
[13/35] Besancon: OK
[14/35] Dijon: OK
[15/35] Annecy: OK
[16/35] Grenoble: OK
[17/35] Lyon: OK
[18/35] Gorges du Verdon: OK
[19/35] Bormes les Mimosas: OK
[20/35] Cassis: OK
[21/35] Marseille: OK
[22/35] Aix en Provence: OK
[23/35] Avignon: OK
[24/35] Uzes: OK
[25/35] Nimes: OK
[26/35] Aigues Mortes: OK
[27/35] Saintes Maries de la mer: OK
[28/35] Collioure: OK
[29/35] Carcassonne: OK
[30/35] Ariege: OK
[31/35] Toulouse: OK
[32/35] Montauban: OK
[33/35] Biarritz: OK
[34/35] Bayonne: OK
[35/35] La Rochelle: OK


,city_id,city_name,query,latitude,longitude,display_name,osm_type,osm_id,class,type,found
0,1,Mont Saint Michel,"Mont Saint Michel, France",48.635954,-1.511460,"Mont Saint-Michel, Le Mont-Saint-Michel, Avran...",way,211285890,None,islet,True
1,2,St Malo,"St Malo, France",48.649518,-2.026041,"Saint-Malo, Ille-et-Vilaine, Bretagne, France ...",relation,905534,None,administrative,True
2,3,Bayeux,"Bayeux, France",49.276462,-0.702474,"Bayeux, Calvados, Normandie, France métropolit...",relation,145776,None,administrative,True
3,4,Le Havre,"Le Havre, France",49.493898,0.107973,"Le Havre, Seine-Maritime, Normandie, France mé...",relation,104492,None,administrative,True
4,5,Rouen,"Rouen, France",49.440459,1.093966,"Rouen, Seine-Maritime, Normandie, France métro...",relation,75628,None,administrative,True


### **Exporter le resultat**

Cette cellule:
- cree le dossier `outputs/data` s'il n'existe pas,
- sauvegarde le DataFrame final dans `cities_geocoded.csv`,
- affiche un recapitulatif rapide du nombre de villes trouvees/non trouvees.

In [5]:
output_dir = Path("../outputs/data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "cities_geocoded.csv"
cities_geo_df.to_csv(output_file, index=False)

print(f"Fichier exporte: {output_file.resolve()}")
print(cities_geo_df["found"].value_counts(dropna=False))

Fichier exporte: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/data/cities_geocoded.csv
found
True    35
Name: count, dtype: int64
